# Quản lý ChromaDB

Notebook này cung cấp các công cụ để quản lý ChromaDB, chủ yếu là xem và xóa dữ liệu.


In [ ]:
# Import thư viện cần thiết
import chromadb
from chromadb.config import Settings
import pandas as pd
from typing import List, Optional
import json

print("Đã import các thư viện cần thiết")


In [ ]:
# Kết nối với ChromaDB
# Thay đổi đường dẫn nếu cần
# CHROMA_DB_PATH = "./chroma_db"  # Đường dẫn đến thư mục chứa ChromaDB

# Kết nối với client
# client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Hoặc nếu sử dụng ChromaDB server:
client = chromadb.HttpClient(host="localhost", port=8000)

# print(f"Đã kết nối với ChromaDB tại: {CHROMA_DB_PATH}")


## 1. Xem danh sách Collections


In [ ]:
# Liệt kê tất cả collections
def list_collections():
    """Liệt kê tất cả collections trong ChromaDB"""
    collections = client.list_collections()
    print(f"\nTổng số collections: {len(collections)}")
    print("\nDanh sách collections:")
    print("-" * 60)
    for i, collection in enumerate(collections, 1):
        print(f"{i}. Tên: {collection.name}")
        print(f"   ID: {collection.id}")
        print(f"   Metadata: {collection.metadata}")
        print()
    return collections

collections = list_collections()


## 2. Xem dữ liệu trong Collection


In [ ]:
# Chọn collection để xem (thay đổi tên collection nếu cần)
COLLECTION_NAME = "phim_collection"  # Thay đổi tên collection ở đây

try:
    collection = client.get_collection(name=COLLECTION_NAME)
    print(f"Đã kết nối với collection: {COLLECTION_NAME}")
except Exception as e:
    print(f"Lỗi: {e}")
    print("Vui lòng kiểm tra lại tên collection hoặc tạo collection mới.")


In [ ]:
# Xem thông tin tổng quan về collection
def get_collection_info(collection):
    """Lấy thông tin tổng quan về collection"""
    count = collection.count()
    print(f"\n{'='*60}")
    print(f"Thông tin Collection: {collection.name}")
    print(f"{'='*60}")
    print(f"Số lượng documents: {count}")
    print(f"Metadata: {collection.metadata}")
    return count

if 'collection' in locals():
    count = get_collection_info(collection)


In [ ]:
# Xem tất cả dữ liệu trong collection (giới hạn số lượng)
def view_collection_data(collection, limit: int = 100):
    """Xem dữ liệu trong collection dưới dạng bảng"""
    try:
        # Lấy tất cả dữ liệu
        results = collection.get(limit=limit)
        
        # Tạo DataFrame để dễ xem
        data = {
            'ID': results['ids'],
            'Documents': results.get('documents', [''] * len(results['ids'])),
            'Metadata': [json.dumps(m, ensure_ascii=False) if m else '' for m in results.get('metadatas', [])]
        }
        
        df = pd.DataFrame(data)
        
        print(f"\n{'='*60}")
        print(f"Dữ liệu trong collection '{collection.name}' (hiển thị {len(df)}/{collection.count()} documents):")
        print(f"{'='*60}\n")
        
        # Hiển thị từng dòng
        for idx, row in df.iterrows():
            print(f"\n[{idx + 1}] ID: {row['ID']}")
            print(f"    Document: {row['Documents'][:100]}..." if len(str(row['Documents'])) > 100 else f"    Document: {row['Documents']}")
            if row['Metadata']:
                print(f"    Metadata: {row['Metadata']}")
            print("-" * 60)
        
        return df
    except Exception as e:
        print(f"Lỗi khi xem dữ liệu: {e}")
        return None

if 'collection' in locals():
    df = view_collection_data(collection, limit=100)


In [ ]:
# Tìm kiếm documents theo ID hoặc metadata
def search_by_id(collection, ids: List[str]):
    """Tìm kiếm documents theo danh sách IDs"""
    try:
        results = collection.get(ids=ids)
        print(f"\nTìm thấy {len(results['ids'])} documents:")
        print("-" * 60)
        for i, doc_id in enumerate(results['ids']):
            print(f"\n[{i+1}] ID: {doc_id}")
            if results.get('documents'):
                print(f"    Document: {results['documents'][i]}")
            if results.get('metadatas') and results['metadatas'][i]:
                print(f"    Metadata: {json.dumps(results['metadatas'][i], ensure_ascii=False)}")
        return results
    except Exception as e:
        print(f"Lỗi: {e}")
        return None

# Ví dụ: Tìm kiếm theo ID
# if 'collection' in locals():
#     search_by_id(collection, ids=["id1", "id2"])


## 3. Xóa dữ liệu


In [ ]:
# Xóa documents theo ID
def delete_documents(collection, ids: List[str], confirm: bool = False):
    """Xóa documents theo danh sách IDs"""
    if not confirm:
        print("⚠️  CHÚ Ý: Bạn cần set confirm=True để thực hiện xóa!")
        print(f"Documents sẽ bị xóa: {ids}")
        return
    
    try:
        # Kiểm tra số lượng trước khi xóa
        before_count = collection.count()
        
        # Xóa documents
        collection.delete(ids=ids)
        
        # Kiểm tra số lượng sau khi xóa
        after_count = collection.count()
        deleted_count = before_count - after_count
        
        print(f"\n✓ Đã xóa thành công {deleted_count} documents")
        print(f"  Số lượng documents trước: {before_count}")
        print(f"  Số lượng documents sau: {after_count}")
        print(f"  IDs đã xóa: {ids}")
    except Exception as e:
        print(f"✗ Lỗi khi xóa documents: {e}")

# Ví dụ: Xóa documents theo ID
# if 'collection' in locals():
#     delete_documents(collection, ids=["id1", "id2"], confirm=True)


In [ ]:
# Xóa documents theo điều kiện (where clause)
def delete_by_metadata(collection, where: dict, confirm: bool = False):
    """Xóa documents theo điều kiện metadata"""
    if not confirm:
        print("⚠️  CHÚ Ý: Bạn cần set confirm=True để thực hiện xóa!")
        print(f"Điều kiện xóa: {where}")
        return
    
    try:
        # Đếm số documents sẽ bị xóa
        results = collection.get(where=where)
        count_before_delete = len(results['ids'])
        
        if count_before_delete == 0:
            print("Không tìm thấy documents nào phù hợp với điều kiện.")
            return
        
        # Xóa documents
        collection.delete(where=where)
        
        print(f"\n✓ Đã xóa thành công {count_before_delete} documents")
        print(f"  Điều kiện: {where}")
    except Exception as e:
        print(f"✗ Lỗi khi xóa documents: {e}")

# Ví dụ: Xóa documents theo metadata
# if 'collection' in locals():
#     delete_by_metadata(collection, where={"category": "test"}, confirm=True)


In [ ]:
# Xóa toàn bộ collection
def delete_collection(collection_name: str, confirm: bool = False):
    """Xóa toàn bộ collection"""
    if not confirm:
        print("⚠️  CHÚ Ý: Bạn cần set confirm=True để thực hiện xóa!")
        print(f"Collection '{collection_name}' sẽ bị xóa hoàn toàn!")
        return
    
    try:
        # Lấy thông tin collection trước khi xóa
        collection = client.get_collection(name=collection_name)
        count = collection.count()
        
        # Xóa collection
        client.delete_collection(name=collection_name)
        
        print(f"\n✓ Đã xóa thành công collection '{collection_name}'")
        print(f"  Số lượng documents đã xóa: {count}")
    except Exception as e:
        print(f"✗ Lỗi khi xóa collection: {e}")

# Ví dụ: Xóa collection
delete_collection("suatchieu_collection", confirm=True)


## 4. Các hàm tiện ích khác


In [ ]:
# Xóa tất cả documents trong collection (giữ lại collection)
def clear_collection(collection, confirm: bool = False):
    """Xóa tất cả documents trong collection nhưng giữ lại collection"""
    if not confirm:
        print("⚠️  CHÚ Ý: Bạn cần set confirm=True để thực hiện xóa!")
        count = collection.count()
        print(f"Tất cả {count} documents trong collection '{collection.name}' sẽ bị xóa!")
        return
    
    try:
        # Lấy tất cả IDs
        all_results = collection.get()
        all_ids = all_results['ids']
        
        if not all_ids:
            print("Collection đã trống.")
            return
        
        # Xóa tất cả
        collection.delete(ids=all_ids)
        
        print(f"\n✓ Đã xóa thành công {len(all_ids)} documents")
        print(f"  Collection '{collection.name}' hiện đã trống")
    except Exception as e:
        print(f"✗ Lỗi khi xóa: {e}")

# Ví dụ: Xóa tất cả documents
# if 'collection' in locals():
#     clear_collection(collection, confirm=True)


In [ ]:
# Xuất dữ liệu collection ra file JSON
def export_collection_to_json(collection, output_file: str = None):
    """Xuất dữ liệu collection ra file JSON"""
    try:
        results = collection.get()
        
        data = {
            'collection_name': collection.name,
            'count': len(results['ids']),
            'documents': []
        }
        
        for i, doc_id in enumerate(results['ids']):
            doc_data = {
                'id': doc_id,
                'document': results.get('documents', [''])[i] if results.get('documents') else '',
                'metadata': results.get('metadatas', [{}])[i] if results.get('metadatas') else {}
            }
            data['documents'].append(doc_data)
        
        if output_file is None:
            output_file = f"{collection.name}_export.json"
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        
        print(f"✓ Đã xuất {len(results['ids'])} documents ra file: {output_file}")
        return output_file
    except Exception as e:
        print(f"✗ Lỗi khi xuất dữ liệu: {e}")
        return None

# Ví dụ: Xuất dữ liệu
# if 'collection' in locals():
#     export_collection_to_json(collection, "backup.json")


## Hướng dẫn sử dụng

1. **Xem danh sách collections**: Chạy cell "list_collections()"
2. **Xem dữ liệu**: 
   - Thay đổi `COLLECTION_NAME` ở cell 6
   - Chạy các cell tiếp theo để xem thông tin và dữ liệu
3. **Xóa dữ liệu**:
   - Xóa theo ID: `delete_documents(collection, ids=["id1", "id2"], confirm=True)`
   - Xóa theo metadata: `delete_by_metadata(collection, where={"key": "value"}, confirm=True)`
   - Xóa tất cả: `clear_collection(collection, confirm=True)`
   - Xóa collection: `delete_collection("collection_name", confirm=True)`

⚠️ **LƯU Ý**: Luôn set `confirm=True` để thực hiện xóa. Mặc định các hàm xóa sẽ không thực thi để tránh xóa nhầm.
